# Optimizer Tests

- Check NanoChat implementation of AdamW/Muon vs PyTorch version
- torch.optim.AdamW and AdamW (tested w/o DDP) are equivalent
- torch.optim.Muon and Muon are equivalent for 2D weight arrays
  + for our purpose in GPT all weights are 2D so we are all good

The conclusion is Karpathy implementation are equivalent to PyTorch for our purposes.

His implementations use less memory due to clever DDP operations.

I don't want to copy-paste code I don't deeply understand, so we will use PyTorch and eat the performance hit for now.

# Test AdamW

In [1]:
import torch
# from nanochat.adamw import DistAdamW

# NOTE: DistAdamW requires DDP initialization
#       Instead we use this modified version. this is basically nanochat DistAdamW
#       with minimal changes (applied by Claude) to remove DDP
#       Original in nanochat/adamw.py git f5a0ea4d3f98be55675d2518a02a7bc3a18236b2
class AdamW_our(torch.optim.Optimizer):
    """
    Single-process AdamW optimizer (DistAdamW without DDP collectives).
    """
    def __init__(self, param_groups, lr: float = 1e-3, betas: tuple[float, float] = (0.9, 0.999), eps: float = 1e-8, weight_decay: float = 0.01):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(param_groups, defaults)

    #@torch.compile
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            beta1, beta2 = group['betas']
            eps = group['eps']
            wd = group['weight_decay']
            params = group['params']
            for base in range(len(params)):
                p = params[base]
                lr = group['lr'] * getattr(p, "lr_mul", 1.0)
                state = self.state[p]
                g_slice = p.grad
                # State init
                if not state:
                    state['step'] = torch.tensor(0, dtype=torch.int64, device=p.device)
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                exp_avg = state['exp_avg']
                exp_avg_sq = state['exp_avg_sq']
                state['step'] += 1
                t = state['step']
                # weight decay
                if wd != 0:
                    eff_weight_decay = lr * wd * getattr(p, "wd_mul", 1.0)
                    p.mul_(1 - eff_weight_decay)
                # update running averages
                exp_avg.mul_(beta1).add_(g_slice, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(g_slice, g_slice, value=1 - beta2)
                # bias corrections
                bias1 = 1 - beta1 ** t
                bias2 = 1 - beta2 ** t
                # compute step
                denom = (exp_avg_sq / bias2).sqrt().add_(eps)
                step_size = lr / bias1
                update = exp_avg.div(denom).mul_(step_size)
                p.add_(other=update, alpha=-1.0)

In [2]:
torch.manual_seed(42)

# Forward + backward
x = torch.randn(16, 32)
target = torch.randn(16, 64)

# Create identical weights
W1 = torch.randn(64, 32, requires_grad=True)
W2 = W1.clone().detach().requires_grad_(True)

In [3]:
# Params
lr = 1e-3
betas = (0.9, 0.999)
eps = 1e-8
weight_decay = 0.01

# Optimizers
opt_torch = torch.optim.AdamW([W1], lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
opt_custom = AdamW_our([{"params": [W2]}], lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)

In [4]:
for i in range(20):
    opt_torch.zero_grad()
    opt_custom.zero_grad()
    loss1 = ((x @ W1.T - target) ** 2).mean()
    loss2 = ((x @ W2.T - target) ** 2).mean()
    loss1.backward()
    loss2.backward()
    opt_torch.step()
    opt_custom.step()

    weight_max_diff = (W1 - W2).abs().max().item()
    # compare optmizer states
    state1 = opt_torch.state[W1]
    state2 = opt_custom.state[W2]
    exp_avg_max_diff = (state1['exp_avg'] - state2['exp_avg']).abs().max().item()
    exp_avg_sq_max_diff = (state1['exp_avg_sq'] - state2['exp_avg_sq']).abs().max().item()
    print(i, weight_max_diff, exp_avg_max_diff, exp_avg_sq_max_diff)

0 1.1920928955078125e-07 0.0 0.0
1 1.1920928955078125e-07 3.725290298461914e-09 3.637978807091713e-12
2 2.384185791015625e-07 3.725290298461914e-09 1.0913936421275139e-11
3 2.384185791015625e-07 5.587935447692871e-09 1.4551915228366852e-11
4 2.384185791015625e-07 7.450580596923828e-09 1.4551915228366852e-11
5 4.76837158203125e-07 1.4901161193847656e-08 2.9103830456733704e-11
6 4.76837158203125e-07 1.4901161193847656e-08 5.820766091346741e-11
7 4.76837158203125e-07 1.4901161193847656e-08 5.820766091346741e-11
8 4.76837158203125e-07 2.2351741790771484e-08 8.731149137020111e-11
9 4.76837158203125e-07 2.9802322387695312e-08 1.1641532182693481e-10
10 4.76837158203125e-07 2.9802322387695312e-08 1.1641532182693481e-10
11 4.76837158203125e-07 2.9802322387695312e-08 1.7462298274040222e-10
12 5.960464477539062e-07 2.9802322387695312e-08 1.7462298274040222e-10
13 5.960464477539062e-07 2.9802322387695312e-08 2.3283064365386963e-10
14 5.960464477539062e-07 2.9802322387695312e-08 2.3283064365386963e

# Test Zeropower via NewtonSchulz

In [5]:
import torch
from torch import Tensor

In [6]:
# Copied from:
# https://raw.githubusercontent.com/pytorch/pytorch/refs/tags/v2.9.1/torch/optim/_muon.py
def _zeropower_via_newtonschulz_pt(
    grad: Tensor, ns_coefficients: tuple[float, float, float], ns_steps: int, eps: float
) -> Tensor:
    """
    Newton-Schulz iteration to compute the zeroth power / orthogonalization of G. We opt to use a
    quintic iteration whose coefficients are selected to maximize the slope at zero. For the purpose
    of minimizing steps, it turns out to be empirically effective to keep increasing the slope at
    zero even beyond the point where the iteration no longer converges all the way to one everywhere
    on the interval. This iteration therefore does not produce UV^T but rather something like US'V^T
    where S' is diagonal with S_{ii}' ~ Uniform(0.5, 1.5), which turns out not to hurt model
    performance at all relative to UV^T, where USV^T = G is the SVD.

    Implementation reference: https://github.com/KellerJordan/Muon/blob/master/muon.py
    with suggestions by @jxbz, @leloykun, and @YouJiacheng.
    """
    if ns_steps >= 100:
        raise ValueError(
            "Number of steps must be less than 100 for computational efficiency"
        )
    if len(grad.shape) != 2:
        raise ValueError("Input tensor gradient must be a 2D matrix")
    if len(ns_coefficients) != 3:
        raise ValueError("Coefficients must be a tuple of exactly 3 values")
    a, b, c = ns_coefficients
    ortho_grad = grad.bfloat16()
    if grad.size(0) > grad.size(1):
        ortho_grad = ortho_grad.T
    # Ensure spectral norm is at most 1
    ortho_grad.div_(ortho_grad.norm().clamp(min=eps))
    # Perform the NS iterations
    for _ in range(ns_steps):
        gram_matrix = ortho_grad @ ortho_grad.T
        gram_update = torch.addmm(
            gram_matrix, gram_matrix, gram_matrix, beta=b, alpha=c
        )
        ortho_grad = torch.addmm(ortho_grad, gram_update, ortho_grad, beta=a)

    if grad.size(0) > grad.size(1):
        ortho_grad = ortho_grad.T
    return ortho_grad

In [7]:
# Copied from nanochat/muon.py git commit f5a0ea4d3f98be55675d2518a02a7bc3a18236b2
def zeropower_via_newtonschulz5(G: Tensor, steps: int) -> Tensor:
    """
    Newton-Schulz iteration to compute the zeroth power / orthogonalization of G. We opt to use a
    quintic iteration whose coefficients are selected to maximize the slope at zero. For the purpose
    of minimizing steps, it turns out to be empirically effective to keep increasing the slope at
    zero even beyond the point where the iteration no longer converges all the way to one everywhere
    on the interval. This iteration therefore does not produce UV^T but rather something like US'V^T
    where S' is diagonal with S_{ii}' ~ Uniform(0.5, 1.5), which turns out not to hurt model
    performance at all relative to UV^T, where USV^T = G is the SVD.
    """
    assert G.ndim >= 2 # batched Muon implementation by @scottjmaddox, and put into practice in the record by @YouJiacheng
    a, b, c = (3.4445, -4.7750,  2.0315)
    X = G.bfloat16()
    if G.size(-2) > G.size(-1):
        X = X.mT

    # Ensure spectral norm is at most 1
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    
    # NOTE: This needs to be patched to get proper equivalence with the reference implementation
    # Perform the NS iterations
    # for _ in range(steps):
    #     A = X @ X.mT
    #     B = b * A + c * A @ A # quintic computation strategy adapted from suggestion by @jxbz, @leloykun, and @YouJiacheng
    #     X = a * X + B @ X
    for _ in range(steps):
        A = X @ X.mT
        B = torch.addmm(A, A, A, beta=b, alpha=c)
        X = torch.addmm(X, B, X, beta=a)


    if G.size(-2) > G.size(-1):
        X = X.mT
    return X

In [8]:
torch.manual_seed(42)
g = torch.randn(64, 32)

out_custom = zeropower_via_newtonschulz5(g.clone(), steps=5)
out_torch = _zeropower_via_newtonschulz_pt(g.clone(), (3.4445, -4.7750, 2.0315), 5, 1e-7)

print("NS output diff:", (out_custom.float() - out_torch.float()).abs().max().item())


NS output diff: 0.0


# Test Muon

In [9]:
import torch

In [10]:
# NOTE: Claude Opus 4.5 claims that this version of Muon is equivalent only for
#       the weigth matrices of 2D shape. Specifically, ours processes batched
#       matrices independently, while pytorch flattens them to calculate norm.
#       I did not investigate deeply, since all matrices in GPT training are 2D.

In [11]:
# copied from nanochat/muon.py git commit f5a0ea4d3f98be55675d2518a02a7bc3a18236b2
class Muon(torch.optim.Optimizer):
    """
    Muon - MomentUm Orthogonalized by Newton-schulz

    https://kellerjordan.github.io/posts/muon/

    Muon internally runs standard SGD-momentum, and then performs an orthogonalization post-
    processing step, in which each 2D parameter's update is replaced with the nearest orthogonal
    matrix. To efficiently orthogonalize each update, we use a Newton-Schulz iteration, which has
    the advantage that it can be stably run in bfloat16 on the GPU.

    Some warnings:
    - This optimizer should not be used for the embedding layer, the final fully connected layer,
    or any {0,1}-D parameters; those should all be optimized by a standard method (e.g., AdamW).
    - To use it with 4D convolutional filters, it works well to just flatten their last 3 dimensions.

    Arguments:
        lr: The learning rate used by the internal SGD.
        momentum: The momentum used by the internal SGD.
        nesterov: Whether to use Nesterov-style momentum in the internal SGD. (recommended)
        ns_steps: The number of Newton-Schulz iteration steps to use.
    """
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)
        params: list[Tensor] = [*params]
        param_groups = []
        for size in {p.numel() for p in params}:
            group = dict(params=[p for p in params if p.numel() == size])
            param_groups.append(group)
        super().__init__(param_groups, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            params: list[Tensor] = group["params"]
            for p in params:
                g = p.grad
                assert g is not None
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf: Tensor = state["momentum_buffer"]
                buf.lerp_(g, 1 - group["momentum"])
                g = g.lerp_(buf, group["momentum"]) if group["nesterov"] else buf
                g = zeropower_via_newtonschulz5(g, steps=group["ns_steps"])
                p.add_(g, alpha=-group["lr"] * max(1, p.size(-2) / p.size(-1))**0.5)

In [12]:
torch.manual_seed(42)

In [13]:
# Forward + backward
x = torch.randn(16, 32)
target = torch.randn(16, 64)

# Create identical weights
W1 = torch.randn(64, 32, requires_grad=True)
W2 = W1.clone().detach().requires_grad_(True)

In [14]:
# NOTE: nanochat _zeropower_via_newtonschulz5 must be patched to match Muon version numerically
#       versions are mathematically equivalent but numerically slightly different

# REPLACE THIS:
# for _ in range(steps):
#     A = X @ X.mT
#     B = b * A + c * A @ A # quintic computation strategy adapted from suggestion by @jxbz, @leloykun, and @YouJiacheng
#     X = a * X + B @ X
# WITH THIS:
# for _ in range(steps):
#     A = X @ X.mT
#     B = torch.addmm(A, A, A, beta=b, alpha=c)
#     X = torch.addmm(X, B, X, beta=a)

In [15]:
# Params
lr = 0.02
momentum = 0.95
nesterov = True
ns_steps = 5

# Optimizers
opt_torch = torch.optim.Muon([W1], lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps,
                             weight_decay=0, adjust_lr_fn="original")
opt_custom = Muon([W2], lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)


In [16]:
for i in range(20):
    opt_torch.zero_grad()
    opt_custom.zero_grad()
    loss1 = ((x @ W1.T - target) ** 2).mean()
    loss2 = ((x @ W2.T - target) ** 2).mean()
    loss1.backward()
    loss2.backward()
    opt_torch.step()
    opt_custom.step()

    weight_max_diff = (W1 - W2).abs().max().item()
    # compare optmizer states
    state1 = opt_torch.state[W1]
    state2 = opt_custom.state[W2]
    momentum_buffer_diff = (state1['momentum_buffer'] - state2['momentum_buffer']).abs().max().item()
    print(i, weight_max_diff, momentum_buffer_diff)

0 0.0 0.0
1 0.0 0.0
2 0.0 0.0
3 0.0 0.0
4 0.0 0.0
5 0.0 0.0
6 0.0 0.0
7 0.0 0.0
8 0.0 0.0
9 0.0 0.0
10 0.0 0.0
11 0.0 0.0
12 0.0 0.0
13 0.0 0.0
14 0.0 0.0
15 0.0 0.0
16 0.0 0.0
17 0.0 0.0
18 0.0 0.0
19 0.0 0.0
